# Preprocessing demo — review → tokens → IDs → padded sequence → tensor

Người phụ trách: **Duy Căn**. Notebook chỉ minh hoạ pipeline tiền xử lý (không train LSTM).

```
Review gốc → Tokenizer → Tokens → Vocabulary → Token IDs → Padding/Truncation → Tensor + Label → Embedding → LSTM
```
Chạy từ thư mục `notebooks/` hoặc thư mục gốc project.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():       # đang đứng trong notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from src.data.tokenizer import Tokenizer
from src.data.vocabulary import Vocabulary
from src.data.padding import pad_or_truncate, effective_length

## 1. Review mẫu

In [ ]:
review = """
I really loved this movie.
The actors were amazing and the story wasn't boring at all!
<br /><br />
Highly recommended &amp; worth watching.
"""
print(review)

## 2. Tokenization
Bỏ thẻ HTML, giải mã `&amp;`, chữ thường, giữ `wasn't` là một token, giữ `!` và `?`.

In [ ]:
tokenizer = Tokenizer()
tokens = tokenizer.tokenize(review)
print(tokens)
print("Số token:", len(tokens))

## 3. Vocabulary
`<PAD>`=0, `<UNK>`=1, token còn lại từ 2 (tần suất giảm dần). Ở đây dùng vài câu mẫu; với dữ liệu thật vocabulary **chỉ build từ tập train** (xem mục 7).

In [ ]:
vocab = Vocabulary()
vocab.build([
    tokens,
    tokenizer.tokenize("This movie was great"),
    tokenizer.tokenize("I really love this film"),
])
print("Vocabulary size:", len(vocab))
print({t: vocab.token_to_id[t] for t in list(vocab.token_to_id)[:12]})

## 4. Token IDs và token ngoài vocabulary (UNK)

In [ ]:
ids = vocab.encode(tokens)
print("Tokens:", tokens)
print("IDs   :", ids)

unseen = tokenizer.tokenize("Terrible acting")      # từ chưa có trong vocabulary
print("\nUnseen:", unseen, "->", vocab.encode(unseen), "(1 = <UNK>)")
print("Decode:", vocab.decode(ids))

## 5. Padding / Truncation

In [ ]:
MAX_LENGTH = 24

padded = pad_or_truncate(ids, max_length=MAX_LENGTH, pad_id=vocab.PAD_ID)
print("Độ dài gốc :", len(ids))
print("Sau padding:", len(padded))
print(padded)

short = pad_or_truncate(ids, max_length=8, pad_id=vocab.PAD_ID)
print("\nTruncate về 8:", short)
print("Độ dài thật (dùng cho pack_padded_sequence):", effective_length(len(ids), MAX_LENGTH))

## 6. Tensor

In [ ]:
input_tensor = torch.tensor(padded, dtype=torch.long)
print(input_tensor)
print(input_tensor.shape)          # torch.Size([24])

batch = input_tensor.unsqueeze(0)  # (batch_size=1, max_length)
print("Batch shape:", tuple(batch.shape), "-> sau Embedding: (1, 24, embedding_dim)")

## 7. (Tuỳ chọn) Chạy trên IMDb thật
Chỉ chạy khi có `data/raw/aclImdb/`. Luồng đúng: tách validation **từ train** → build vocabulary **chỉ từ train** → dùng lại vocabulary cho validation/test.

In [ ]:
from src.data.dataset import (
    IMDBDataset, collect_samples, split_train_val,
    build_vocabulary, preprocessing_stats, create_dataloader,
)

IMDB_DIR = ROOT / "data" / "raw" / "aclImdb"
SEED, MAX_LENGTH, VAL_RATIO = 42, 200, 0.2    # max_length: chốt theo EDA của Quốc Anh

if not IMDB_DIR.exists():
    print("Chưa có", IMDB_DIR, "- bỏ qua mục này.")
else:
    all_train = collect_samples(IMDB_DIR / "train")
    train_s, val_s = split_train_val(all_train, VAL_RATIO, SEED)
    test_s = collect_samples(IMDB_DIR / "test")
    print("train / val / test:", len(train_s), len(val_s), len(test_s))

    real_vocab = build_vocabulary(train_s, tokenizer, min_freq=2)   # CHỈ train
    print("Vocabulary size:", len(real_vocab))

    for name, split in [("train", train_s), ("val", val_s), ("test", test_s)]:
        print(name, preprocessing_stats(split, tokenizer, real_vocab, MAX_LENGTH))

    train_ds = IMDBDataset(tokenizer, real_vocab, MAX_LENGTH, samples=train_s)
    x, y = train_ds[0]
    print("Một mẫu:", x.shape, y)

    xb, yb = next(iter(create_dataloader(train_ds, batch_size=32, shuffle=True, seed=SEED)))
    print("Batch:", xb.shape, yb.shape)     # (32, 200) (32,)